In [4]:
import numpy as np

num_states = 5
noise = 0.5
A_1 = np.eye(num_states)

print(A_1)

[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]


In [8]:
def obs_tensor_via_kron(n: float) -> np.ndarray:
    S = np.array([[1 - n, n],
                [n, 1 - n]], dtype=float)
    K = np.kron(S, S)                  # 4x4: P(obs_pair | state_pair)
    return K.reshape(4, 2, 2)          # [obs, s1, s2] with obs=[CC,CD,DC,DD]
A_1 = obs_tensor_via_kron(0.3)
A_1[0, 0, 0]

np.float64(0.48999999999999994)

In [16]:
def noisy_obs_matrix(n: float, P0: np.ndarray | None = None) -> np.ndarray:
    """
    Returns P_noisy(o|s) given noise n and a base emission matrix P0(o|s).
    Convention: rows=observations, cols=true states; each column sums to 1.
    If P0 is None, uses the identity (no base noise).
    """
    S = np.array([[1 - n, n],
                  [n, 1 - n]], dtype=float)        # single-player BSC(n)
    K = np.kron(S, S)                               # two players: independent noise
    if P0 is None:
        return K
    return K @ P0                                   # compose channels

P0 = np.eye(num_states-1)
P_noisy = noisy_obs_matrix(noise, P0)
print(P_noisy)


[[0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]]


In [14]:
A_1[1:,1:] = P_noisy
print(A_1)
assert  A_1.sum(axis=1, keepdims=True).all() == 1

[[1.   0.   0.   0.   0.  ]
 [0.   0.81 0.09 0.09 0.01]
 [0.   0.09 0.81 0.01 0.09]
 [0.   0.09 0.01 0.81 0.09]
 [0.   0.01 0.09 0.09 0.81]]


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10


In [ ]:
# Data loading functions
def load_round_robin_data(base_dir="/home/kivalm/code/AIFNIPD/experimentation/round_robin"):
    """
    Load all round robin tournament data from CSV and JSON files
    """
    base_path = Path(base_dir)
    
    # Get all repetition directories
    repetition_dirs = [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith('repetition_')]
    repetition_dirs.sort(key=lambda x: int(x.name.split('_')[1]))
    
    print(f"Found {len(repetition_dirs)} repetition directories")
    
    # Get noise levels from first repetition
    sample_rep = repetition_dirs[0]
    noise_levels = sorted([float(f.stem) for f in sample_rep.iterdir() if f.suffix == '.csv'])
    print(f"Noise levels: {noise_levels}")
    
    # Load player names from details.json
    details_file = sample_rep / 'details.json'
    with open(details_file, 'r') as f:
        details = json.load(f)
    player_names = details['player_names']
    print(f"Found {len(player_names)} players")
    
    all_data = {}
    
    # Load data for each noise level
    for noise_level in tqdm(noise_levels, desc="Loading noise levels"):
        noise_data = {
            'csv_data': [],
            'json_data': []
        }
        
        # Load data from all repetitions for this noise level
        for rep_dir in repetition_dirs:
            csv_file = rep_dir / f"{noise_level}.csv"
            json_file = rep_dir / f"{noise_level}.json"
            
            if csv_file.exists() and json_file.exists():
                # Load CSV data
                df = pd.read_csv(csv_file)
                df['repetition_id'] = int(rep_dir.name.split('_')[1])
                df['noise_level'] = noise_level
                noise_data['csv_data'].append(df)
                
                # Load JSON data
                with open(json_file, 'r') as f:
                    json_data = json.load(f)
                    json_data['repetition_id'] = int(rep_dir.name.split('_')[1])
                    json_data['noise_level'] = noise_level
                    noise_data['json_data'].append(json_data)
        
        # Combine all repetitions for this noise level
        if noise_data['csv_data']:
            all_data[noise_level] = {
                'csv': pd.concat(noise_data['csv_data'], ignore_index=True),
                'json': noise_data['json_data']
            }
    
    return all_data, player_names


In [ ]:
def calculate_cooperation_rate(actions_str):
    """Calculate cooperation rate from action string"""
    if pd.isna(actions_str) or actions_str == '':
        return 0.0
    return actions_str.count('C') / len(actions_str)

def calculate_mutual_cooperation_rate(actions1, actions2):
    """Calculate mutual cooperation rate between two action strings"""
    if pd.isna(actions1) or pd.isna(actions2) or actions1 == '' or actions2 == '':
        return 0.0
    
    min_len = min(len(actions1), len(actions2))
    mutual_coop_count = sum(1 for i in range(min_len) if actions1[i] == 'C' and actions2[i] == 'C')
    return mutual_coop_count / min_len if min_len > 0 else 0.0

def analyze_cooperation_and_scores(all_data, player_names):
    """
    Analyze cooperation rates and scores across noise levels
    """
    results = {}
    
    for noise_level in all_data.keys():
        df = all_data[noise_level]['csv']
        
        # Calculate cooperation rates using intended actions (prenoise)
        coop_rates = {}
        mutual_coop_rates = {}
        score_matrix = np.zeros((len(player_names), len(player_names)))
        
        # Group by interaction to get pairs of players
        for interaction_idx in df['Interaction index'].unique():
            interaction_data = df[df['Interaction index'] == interaction_idx]
            
            if len(interaction_data) == 2:  # Should be exactly 2 players per interaction
                player1_data = interaction_data.iloc[0]
                player2_data = interaction_data.iloc[1]
                
                p1_idx = player1_data['Player index']
                p2_idx = player1_data['Opponent index']
                p1_name = player1_data['Player name']
                p2_name = player1_data['Opponent name']
                
                # Get intended actions (prenoise)
                p1_intended = player1_data['Intended Actions']
                p2_intended = player2_data['Intended Actions']
                
                # Calculate cooperation rates
                p1_coop = calculate_cooperation_rate(p1_intended)
                p2_coop = calculate_cooperation_rate(p2_intended)
                
                # Calculate mutual cooperation rate
                mutual_coop = calculate_mutual_cooperation_rate(p1_intended, p2_intended)
                
                # Store cooperation rates
                if p1_name not in coop_rates:
                    coop_rates[p1_name] = []
                if p2_name not in coop_rates:
                    coop_rates[p2_name] = []
                
                coop_rates[p1_name].append(p1_coop)
                coop_rates[p2_name].append(p2_coop)
                
                # Store mutual cooperation rate
                pair_key = tuple(sorted([p1_name, p2_name]))
                if pair_key not in mutual_coop_rates:
                    mutual_coop_rates[pair_key] = []
                mutual_coop_rates[pair_key].append(mutual_coop)
                
                # Store scores in matrix
                score_matrix[p1_idx, p2_idx] += player1_data['Score']
                score_matrix[p2_idx, p1_idx] += player2_data['Score']
        
        # Average cooperation rates across all games for each player
        avg_coop_rates = {name: np.mean(rates) for name, rates in coop_rates.items()}
        
        # Average mutual cooperation rates
        avg_mutual_coop = {pair: np.mean(rates) for pair, rates in mutual_coop_rates.items()}
        
        # Average scores (normalize by number of games)
        num_reps = len(df['repetition_id'].unique())
        avg_score_matrix = score_matrix / num_reps
        
        results[noise_level] = {
            'cooperation_rates': avg_coop_rates,
            'mutual_cooperation_rates': avg_mutual_coop,
            'score_matrix': avg_score_matrix,
            'raw_coop_data': coop_rates,
            'raw_mutual_coop_data': mutual_coop_rates
        }
    
    return results


In [ ]:
def plot_cooperation_vs_noise(analysis_results, player_names):
    """
    Plot average cooperation rates vs noise levels for each player
    """
    noise_levels = sorted(analysis_results.keys())
    
    # Prepare data for plotting
    coop_data = []
    for noise in noise_levels:
        coop_rates = analysis_results[noise]['cooperation_rates']
        for player in player_names:
            if player in coop_rates:
                coop_data.append({
                    'noise_level': noise,
                    'player': player,
                    'cooperation_rate': coop_rates[player]
                })
    
    coop_df = pd.DataFrame(coop_data)
    
    # Create the plot
    plt.figure(figsize=(14, 10))
    
    # Plot individual player lines
    for player in player_names:
        player_data = coop_df[coop_df['player'] == player]
        if not player_data.empty:
            plt.plot(player_data['noise_level'], player_data['cooperation_rate'], 
                    label=player, marker='o', alpha=0.7)
    
    plt.xlabel('Noise Level')
    plt.ylabel('Average Cooperation Rate (Intended Actions)')
    plt.title('Player Cooperation Rates vs Noise Level')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return coop_df

def plot_average_mutual_cooperation(analysis_results):
    """
    Plot average mutual cooperation rate vs noise levels
    """
    noise_levels = sorted(analysis_results.keys())
    avg_mutual_coop = []
    
    for noise in noise_levels:
        mutual_rates = list(analysis_results[noise]['mutual_cooperation_rates'].values())
        if mutual_rates:
            avg_mutual_coop.append(np.mean(mutual_rates))
        else:
            avg_mutual_coop.append(0.0)
    
    plt.figure(figsize=(10, 6))
    plt.plot(noise_levels, avg_mutual_coop, marker='o', linewidth=2, markersize=8)
    plt.xlabel('Noise Level')
    plt.ylabel('Average Mutual Cooperation Rate')
    plt.title('Average Mutual Cooperation Rate vs Noise Level')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return noise_levels, avg_mutual_coop

def plot_score_heatmaps(analysis_results, player_names, max_noise_levels=3):
    """
    Plot score heatmaps for different noise levels
    """
    noise_levels = sorted(analysis_results.keys())
    selected_noise = noise_levels[::len(noise_levels)//max_noise_levels][:max_noise_levels]
    
    fig, axes = plt.subplots(1, len(selected_noise), figsize=(6*len(selected_noise), 8))
    if len(selected_noise) == 1:
        axes = [axes]
    
    for i, noise in enumerate(selected_noise):
        score_matrix = analysis_results[noise]['score_matrix']
        
        # Create heatmap
        im = axes[i].imshow(score_matrix, cmap='RdYlBu_r', aspect='auto')
        
        # Set labels
        axes[i].set_title(f'Average Scores (Noise = {noise})')
        axes[i].set_xlabel('Opponent')
        axes[i].set_ylabel('Player')
        
        # Set tick labels (rotate for readability)
        axes[i].set_xticks(range(len(player_names)))
        axes[i].set_yticks(range(len(player_names)))
        axes[i].set_xticklabels(player_names, rotation=45, ha='right')
        axes[i].set_yticklabels(player_names)
        
        # Add colorbar
        plt.colorbar(im, ax=axes[i], shrink=0.8)
    
    plt.tight_layout()
    plt.show()

def plot_player_performance_summary(analysis_results, player_names):
    """
    Create summary plots showing player performance across noise levels
    """
    noise_levels = sorted(analysis_results.keys())
    
    # Calculate average scores for each player at each noise level
    player_avg_scores = {}
    for player in player_names:
        player_avg_scores[player] = []
        for noise in noise_levels:
            score_matrix = analysis_results[noise]['score_matrix']
            player_idx = player_names.index(player)
            # Average score against all opponents
            avg_score = np.mean(score_matrix[player_idx, :])
            player_avg_scores[player].append(avg_score)
    
    # Create subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    
    # Plot 1: Average scores vs noise
    for player in player_names:
        ax1.plot(noise_levels, player_avg_scores[player], 
                label=player, marker='o', alpha=0.7)
    
    ax1.set_xlabel('Noise Level')
    ax1.set_ylabel('Average Score')
    ax1.set_title('Player Average Scores vs Noise Level')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Score difference from no-noise baseline
    baseline_scores = {player: player_avg_scores[player][0] for player in player_names}
    
    for player in player_names:
        score_diffs = [score - baseline_scores[player] for score in player_avg_scores[player]]
        ax2.plot(noise_levels, score_diffs, 
                label=player, marker='o', alpha=0.7)
    
    ax2.set_xlabel('Noise Level')
    ax2.set_ylabel('Score Difference from No-Noise Baseline')
    ax2.set_title('Score Change Due to Noise')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    return player_avg_scores


In [ ]:
# Load and analyze the data
print("Loading round robin tournament data...")
all_data, player_names = load_round_robin_data()

print(f"\nLoaded data for {len(all_data)} noise levels")
print(f"Players: {len(player_names)}")
print(f"Noise levels: {sorted(all_data.keys())}")

# Analyze cooperation rates and scores
print("\nAnalyzing cooperation rates and scores...")
analysis_results = analyze_cooperation_and_scores(all_data, player_names)

print("Analysis complete!")
print(f"Results available for noise levels: {sorted(analysis_results.keys())}")


In [ ]:
# Plot 1: Individual player cooperation rates vs noise
print("Creating cooperation rate plots...")
coop_df = plot_cooperation_vs_noise(analysis_results, player_names)


In [ ]:
# Plot 2: Average mutual cooperation rate vs noise
print("Creating mutual cooperation plot...")
noise_levels, avg_mutual_coop = plot_average_mutual_cooperation(analysis_results)


In [ ]:
# Plot 3: Score heatmaps for selected noise levels
print("Creating score heatmaps...")
plot_score_heatmaps(analysis_results, player_names, max_noise_levels=3)


In [ ]:
# Plot 4: Player performance summary
print("Creating performance summary plots...")
player_avg_scores = plot_player_performance_summary(analysis_results, player_names)


In [ ]:
# Summary statistics and insights
print("=== SUMMARY STATISTICS ===\n")

# Top performers at different noise levels
noise_levels = sorted(analysis_results.keys())
print("Top 3 performers by average score at each noise level:")
for noise in [0.0, 0.25, 0.5]:  # Show key noise levels
    if noise in analysis_results:
        print(f"\nNoise level {noise}:")
        player_scores = []
        for i, player in enumerate(player_names):
            avg_score = np.mean(analysis_results[noise]['score_matrix'][i, :])
            player_scores.append((player, avg_score))
        
        player_scores.sort(key=lambda x: x[1], reverse=True)
        for rank, (player, score) in enumerate(player_scores[:3], 1):
            print(f"  {rank}. {player}: {score:.1f}")

# Cooperation rates at no noise
print(f"\nCooperation rates at no noise (noise = 0.0):")
if 0.0 in analysis_results:
    coop_rates = analysis_results[0.0]['cooperation_rates']
    sorted_coop = sorted(coop_rates.items(), key=lambda x: x[1], reverse=True)
    for player, rate in sorted_coop[:5]:
        print(f"  {player}: {rate:.3f}")

# Effect of noise on mutual cooperation
print(f"\nEffect of noise on overall mutual cooperation:")
for noise in [0.0, 0.1, 0.25, 0.5]:
    if noise in analysis_results:
        mutual_rates = list(analysis_results[noise]['mutual_cooperation_rates'].values())
        if mutual_rates:
            avg_rate = np.mean(mutual_rates)
            print(f"  Noise {noise}: {avg_rate:.3f}")

print(f"\nAnalysis complete! Data covers {len(player_names)} players across {len(noise_levels)} noise levels.")
